In [2]:
import pandas as pd
import glob

# 앞 노트북에서 만든 타깃 완성본
base = pd.read_csv('target_base.csv')
base['date'] = pd.to_datetime(base['date'])

# 날씨, 미세먼지 (DB 추출 최신본)
weather = pd.read_csv(sorted(glob.glob('../weather_hourly_2*.csv'))[-1])
air     = pd.read_csv(sorted(glob.glob('../air_quality_hourly_2*.csv'))[-1])

weather['ts_date'] = pd.to_datetime(weather['ts_date'])
air['ts_date']     = pd.to_datetime(air['ts_date'])

# 날씨 중복 제거: 2025-07-02부터 같은 시간이 이중 저장되어 있음 (값은 동일)
# (ts_date, hour) 조합당 한 행만 남긴다
weather = weather.drop_duplicates(subset=['ts_date', 'hour'])

print(f"base {len(base):,}행 / weather {len(weather):,}행 / air {len(air):,}행")

base 176,928행 / weather 12,816행 / air 10,176행


In [3]:
# 필요한 컬럼만 골라서 (date, hour) 키로 left join
# left join이므로 base의 행수는 절대 변하면 안 됨 (변하면 키 중복이 있다는 뜻)
wx_cols = weather[['ts_date', 'hour', 'temp', 'humid', 'wind', 'precip']]

df = base.merge(
    wx_cols,
    left_on=['date', 'hour'],
    right_on=['ts_date', 'hour'],
    how='left'
).drop(columns='ts_date')   # 조인용으로만 쓴 중복 날짜컬럼 제거

print(f"조인 후 {len(df):,}행 (base와 같아야 함: {len(base):,})")
print(f"날씨 결측: {df['temp'].isna().sum()}행")
print(df.head(5).to_string())

조인 후 176,928행 (base와 같아야 함: 176,928)
날씨 결측: 1881행
  station_id       date  hour  rent_cnt  return_cnt  temp  humid  wind  precip
0     ST0685 2025-01-01     0         0           0 -2.19  59.33  0.91     0.0
1     ST0685 2025-01-01     1         0           0 -2.56  62.22  0.74     0.0
2     ST0685 2025-01-01     2         0           0 -2.80  64.76  0.69     0.0
3     ST0685 2025-01-01     3         0           0 -2.88  66.23  0.62     0.0
4     ST0685 2025-01-01     4         0           0 -3.28  69.10  0.82     0.0


In [4]:
# 같은 방식. no2/co/so2는 1차 모델에서 안 쓰기로 했으니 3개만
aq_cols = air[['ts_date', 'hour', 'pm10', 'pm25', 'ozone']]

df = df.merge(
    aq_cols,
    left_on=['date', 'hour'],
    right_on=['ts_date', 'hour'],
    how='left'
).drop(columns='ts_date')

print(f"조인 후 {len(df):,}행")
print(f"미세먼지 결측: pm10 {df['pm10'].isna().sum()}행 / pm25 {df['pm25'].isna().sum()}행")
print(df.head(5).to_string())

조인 후 176,928행
미세먼지 결측: pm10 2527행 / pm25 6536행
  station_id       date  hour  rent_cnt  return_cnt  temp  humid  wind  precip  pm10  pm25   ozone
0     ST0685 2025-01-01     0         0           0 -2.19  59.33  0.91     0.0   NaN   NaN     NaN
1     ST0685 2025-01-01     1         0           0 -2.56  62.22  0.74     0.0  20.0  13.0  0.0058
2     ST0685 2025-01-01     2         0           0 -2.80  64.76  0.69     0.0  23.0  17.0  0.0046
3     ST0685 2025-01-01     3         0           0 -2.88  66.23  0.62     0.0  22.0  13.0  0.0030
4     ST0685 2025-01-01     4         0           0 -3.28  69.10  0.82     0.0  24.0   7.0  0.0063


In [5]:
env_cols = ['temp', 'humid', 'wind', 'precip', 'pm10', 'pm25', 'ozone']

print("컬럼별 결측:")
print(df[env_cols].isna().sum().to_string())
print(f"\n전체 {len(df):,}행 중 결측이 하나라도 있는 행: {df[env_cols].isna().any(axis=1).sum():,}행")

컬럼별 결측:
temp      1881
humid     1957
wind      1881
precip    1881
pm10      2527
pm25      6536
ozone     3876

전체 176,928행 중 결측이 하나라도 있는 행: 10,317행


In [6]:
# 시간 순서대로 보간해야 하므로 정렬 먼저
df = df.sort_values(['station_id', 'date', 'hour']).reset_index(drop=True)

# 스테이션별로 묶어서 보간 (각 스테이션이 같은 시계열을 갖고 있으므로 결과는 동일하지만,
# 스테이션 경계를 넘어 엉뚱한 값끼리 이어지는 걸 막기 위해 groupby 필수)
df[env_cols] = (
    df.groupby('station_id')[env_cols]
      .transform(lambda s: s.interpolate(method='linear', limit_direction='both'))
)
# limit_direction='both': 맨 앞/맨 뒤 결측(2025-01-01 0시 미세먼지 등)도 가장 가까운 값으로 채움

print("보간 후 결측:")
print(df[env_cols].isna().sum().to_string())   # 전부 0이어야 함

보간 후 결측:
temp      0
humid     0
wind      0
precip    0
pm10      0
pm25      0
ozone     0


In [7]:
# 원래 pm10이 비어있던 시간대 하나 골라서 앞뒤와 자연스럽게 이어졌는지 확인
# (셀 4에서 결측이 있던 날짜로 바꿔서 보세요)
check = df[(df['station_id']=='ST0065') & (df['date']=='2025-01-01')][['date','hour','temp','pm10','pm25']]
print(check.head(12).to_string())

df.to_csv('feature_base.csv', index=False)
print(f"\n저장: feature_base.csv ({len(df):,}행)")

           date  hour  temp  pm10  pm25
9312 2025-01-01     0 -2.19  20.0  13.0
9313 2025-01-01     1 -2.56  20.0  13.0
9314 2025-01-01     2 -2.80  23.0  17.0
9315 2025-01-01     3 -2.88  22.0  13.0
9316 2025-01-01     4 -3.28  24.0   7.0
9317 2025-01-01     5 -3.36  14.0   7.0
9318 2025-01-01     6 -3.50  11.0   7.0
9319 2025-01-01     7 -3.65  12.0   6.0
9320 2025-01-01     8 -2.95  17.0   6.0
9321 2025-01-01     9 -0.25  14.0   2.0
9322 2025-01-01    10  2.53  13.0  12.0
9323 2025-01-01    11  5.87  21.0  19.0

저장: feature_base.csv (176,928행)


In [8]:
# date에서 뽑아낼 수 있는 것들
df['dow'] = df['date'].dt.dayofweek            # 요일 (0=월 ~ 6=일)
df['is_weekend'] = (df['dow'] >= 5).astype(int)  # 토(5), 일(6)이면 1
df['month'] = df['date'].dt.month              # 월 (계절성)

print(df[['date', 'hour', 'dow', 'is_weekend', 'month']].head(5).to_string())

        date  hour  dow  is_weekend  month
0 2025-01-01     0    2           0      1
1 2025-01-01     1    2           0      1
2 2025-01-01     2    2           0      1
3 2025-01-01     3    2           0      1
4 2025-01-01     4    2           0      1


In [9]:
# 학습 기간(2025-01 ~ 2026-02) 안의 법정공휴일 + 임시공휴일 + 대체공휴일
# EDA에서 봤듯 명절엔 수요 패턴이 평일과 완전히 달라서 중요한 피처
holidays = pd.to_datetime([
    # 2025년
    '2025-01-01',                                # 신정
    '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30',  # 설연휴+임시공휴일
    '2025-03-01', '2025-03-03',                  # 삼일절+대체
    '2025-05-05', '2025-05-06',                  # 어린이날/석가탄신일+대체
    '2025-06-03',                                # 대선일(임시공휴일)
    '2025-06-06',                                # 현충일
    '2025-08-15',                                # 광복절
    '2025-10-03', '2025-10-05', '2025-10-06', '2025-10-07', '2025-10-08', '2025-10-09',  # 개천절+추석연휴+대체+한글날
    '2025-12-25',                                # 크리스마스 (12월은 데이터 없지만 리스트엔 유지)
    # 2026년
    '2026-01-01',                                # 신정
    '2026-02-16', '2026-02-17', '2026-02-18',    # 설연휴
])

df['is_holiday'] = df['date'].isin(holidays).astype(int)

# 확인: 공휴일로 표시된 날이 며칠인지
print(f"공휴일 일수: {df[df['is_holiday']==1]['date'].nunique()}일")
print(df[df['is_holiday']==1]['date'].dt.date.unique())

공휴일 일수: 20일
[datetime.date(2025, 1, 1) datetime.date(2025, 1, 27)
 datetime.date(2025, 1, 28) datetime.date(2025, 1, 29)
 datetime.date(2025, 1, 30) datetime.date(2025, 5, 5)
 datetime.date(2025, 5, 6) datetime.date(2025, 6, 3)
 datetime.date(2025, 6, 6) datetime.date(2025, 8, 15)
 datetime.date(2025, 10, 3) datetime.date(2025, 10, 5)
 datetime.date(2025, 10, 6) datetime.date(2025, 10, 7)
 datetime.date(2025, 10, 8) datetime.date(2025, 10, 9)
 datetime.date(2026, 1, 1) datetime.date(2026, 2, 16)
 datetime.date(2026, 2, 17) datetime.date(2026, 2, 18)]


In [10]:
# 최종 컬럼 구성 확인
print(f"총 {len(df):,}행 × {df.shape[1]}컬럼")
print(df.dtypes.to_string())
print()
print(df.head(3).to_string())

# 최종 학습 데이터셋 저장
df.to_csv('tashu_ml_dataset_v1.csv', index=False)
print("\n저장: tashu_ml_dataset_v1.csv")

총 176,928행 × 16컬럼
station_id               str
date          datetime64[us]
hour                   int64
rent_cnt               int64
return_cnt             int64
temp                 float64
humid                float64
wind                 float64
precip               float64
pm10                 float64
pm25                 float64
ozone                float64
dow                    int32
is_weekend             int64
month                  int32
is_holiday             int64

  station_id       date  hour  rent_cnt  return_cnt  temp  humid  wind  precip  pm10  pm25   ozone  dow  is_weekend  month  is_holiday
0     ST0056 2025-01-01     0         0           0 -2.19  59.33  0.91     0.0  20.0  13.0  0.0058    2           0      1           1
1     ST0056 2025-01-01     1         0           0 -2.56  62.22  0.74     0.0  20.0  13.0  0.0058    2           0      1           1
2     ST0056 2025-01-01     2         0           0 -2.80  64.76  0.69     0.0  23.0  17.0  0.0046    2         

In [11]:
# 시간 기준 분할 (아까 정한 경계)
train = df[df['date'] <= '2025-11-30']
val   = df[(df['date'] >= '2026-01-01') & (df['date'] <= '2026-01-31')]
test  = df[df['date'] >= '2026-02-01']

print(f"train: {len(train):,}행 ({train['date'].min().date()} ~ {train['date'].max().date()})")
print(f"val:   {len(val):,}행 ({val['date'].min().date()} ~ {val['date'].max().date()})")
print(f"test:  {len(test):,}행 ({test['date'].min().date()} ~ {test['date'].max().date()})")

train.to_csv('train.csv', index=False)
val.to_csv('val.csv', index=False)
test.to_csv('test.csv', index=False)

train: 150,024행 (2025-01-01 ~ 2025-11-30)
val:   14,136행 (2026-01-01 ~ 2026-01-31)
test:  12,768행 (2026-02-01 ~ 2026-02-28)
